# Torch2 Prediction Notebook

Load the latest saved V2 checkpoint for a model prefix and test moves on a board.

## Configuration and imports

Keep `model_name` aligned with the prefix used during training.

In [ ]:
import torch
from chess import Board, pgn

from auxiliary_func_v2 import (
    DEFAULT_MODEL_NAME,
    DEFAULT_MODELS_DIR,
    board_to_matrix_v2,
    find_latest_training_artifacts,
    load_move_map_data,
)
from model_v2 import ChessModelV2

CONFIG = {
    'model_name': DEFAULT_MODEL_NAME,
    'models_dir': DEFAULT_MODELS_DIR,
}

CONFIG


## Load the latest saved artifacts

The helper scans `../../models/` and picks the newest dated checkpoint that matches the configured prefix.

In [ ]:
artifacts = find_latest_training_artifacts(
    model_name=CONFIG['model_name'],
    models_dir=CONFIG['models_dir'],
)
move_map_data = load_move_map_data(artifacts['map_path'])
move_to_int = move_map_data['move_to_int']
int_to_move = move_map_data['int_to_move']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChessModelV2(num_classes=len(move_to_int))
model.load_state_dict(torch.load(artifacts['model_path'], map_location=device))
model.to(device)
model.eval()

print(f'Using device: {device}')
print(f"Loaded artifact prefix: {artifacts['artifact_prefix']}")
print(f"Trained on: {move_map_data['trained_on']}")
print(f"Model path: {artifacts['model_path']}")
print(f"Map path: {artifacts['map_path']}")


## Prediction helpers

The policy head produces logits over encoded moves; the helper below filters them to legal moves only.

In [ ]:
def prepare_input(board: Board) -> torch.Tensor:
    matrix = board_to_matrix_v2(board)
    return torch.tensor(matrix, dtype=torch.float32).unsqueeze(0)


def predict_move(board: Board, top_k: int = 5):
    x_tensor = prepare_input(board).to(device)

    with torch.no_grad():
        policy_logits, value_pred = model(x_tensor)

    probabilities = torch.softmax(policy_logits.squeeze(0), dim=0).cpu().numpy()
    legal_moves = list(board.legal_moves)
    ranked_candidates = []

    for move in legal_moves:
        move_idx = move_to_int.get(move.uci())
        if move_idx is None:
            continue
        ranked_candidates.append((float(probabilities[move_idx]), move))

    ranked_candidates.sort(key=lambda item: item[0], reverse=True)

    if ranked_candidates:
        best_move = ranked_candidates[0][1]
    else:
        best_move = legal_moves[0] if legal_moves else None

    top_predictions = [(move.uci(), prob) for prob, move in ranked_candidates[:top_k]]
    return best_move, top_predictions, float(value_pred.item())


## Try a board position

In [ ]:
board = Board()
board


In [ ]:
board.push_uci('e2e4')
best_move, top_predictions, position_value = predict_move(board)

print(f'Predicted move: {best_move}')
print(f'Value head estimate: {position_value:.4f}')
print('Top legal candidates:', top_predictions)

if best_move is not None:
    board.push(best_move)

board


In [ ]:
print(str(pgn.Game.from_board(board)))
